[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation.ipynb)

# 04. `rag-regulation-example` 동행 노트북

> 대상 프로젝트: [`example-projects/rag-regulation-example`](../../../example-projects/rag-regulation-example) (C파트: 검색/응답)
> · 앞 단계: [`02_preprocess`](../02_preprocess/02_preprocess.ipynb), [`03_document_input`](../03_document_input/03_document_input.ipynb)
> · 다른 선택지: [ALTERNATIVES.md](../../../example-projects/rag-regulation-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

01이 문서를 모으고, 02가 검색 가능하게 만들고, 03이 사용자 입력을 질문으로 바꿨습니다.
이 프로젝트가 **실제로 답을 내놓습니다.** 여기가 종착점입니다.

그런데 앞의 02에서 500자 고정 길이로 잘랐던 걸 기억하시나요?
이 프로젝트도 원래 1000자를 썼습니다. 그러면 이런 일이 생깁니다.

```
청크 A: ...제11조(재택근무) ① 사원은 부서장의 승인을 받아 재택근무를 할 수 있다. ② 재택근무일의 소정근로시
청크 B: 간은 제9조 제2항을 준용한다. ③ 재택근무 중 발생한 연장근로에 대하여는 제12조에 따른 수당을 지급한다.
```

"재택근무 중 야근수당"의 답은 청크 B에 있습니다. 그런데 **청크 B에는 "재택근무"라는 단어도,
조항 번호도 없습니다.** 검색에 안 걸리고, 걸려도 AI가 "제11조 ③항에 따르면"이라고 말할 수 없습니다.

**규정 문서에는 사람이 이미 정해둔 경계가 있습니다. 조항이죠.**
자로 재서 자르는 대신 그 경계를 쓰면 됩니다. 이 노트북은 그 차이를 직접 재현해보고,
고친 뒤 **정말 나아졌는지 숫자로 확인**하는 데까지 갑니다.

```
규정 PDF -> 구조 파싱 -> 조항 단위 청킹 -> 임베딩 -> OpenSearch
질문 -> 벡터 검색 + 키워드 검색 -> RRF 병합 (20개)
     -> 리랭킹 (4개) -> 프롬프트 조립 -> LLM 답변
평가 -> 골든셋으로 hit@5 / MRR 채점
```

```
01 crawl-storage → 02 preprocess → 03 document-input → [04 rag-regulation]
                                                             여기
```

이번 장에서 배우는 것

- 고정 길이 청킹이 규정 문서에서 실패하는 과정을 **직접 재현**
- 정규식으로 장>절>조>항 구조를 읽고 [조항 단위 청킹](../../../glossary.md#article-chunking)하기
- 조 번호 연속성으로 **PDF 추출의 조용한 실패**를 잡아내는 무결성 검증
- [벡터 검색](../../../glossary.md#vector-search) + [키워드 검색](../../../glossary.md#keyword-search)을 [RRF](../../../glossary.md#rrf)로 합치는 이유
- [리랭커](../../../glossary.md#reranker)(cross-encoder)가 임베딩과 무엇이 다르고 왜 2단으로 쓰는지
- [골든셋](../../../glossary.md#golden-set)과 [hit@k / MRR](../../../glossary.md#retrieval-metrics)로 검색 품질을 채점하기
- **그리고 그 지표가 고장 났을 때 알아채는 법** (실제로 겪습니다)

**소요 시간**: 60~70분. 이 시리즈에서 가장 깁니다.
**[OpenSearch](../../../glossary.md#opensearch)도 API 키도 없이 끝까지 실행됩니다.**
검색은 TF-IDF로 대체하고, 리랭커는 선택적으로 켤 수 있게 해뒀습니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 앞의 01~03을 안 봤어도 따라갈 수 있게 썼습니다. 02를 봤다면 5번이 더 잘 와닿습니다.
- 7번의 리랭커 셀은 기본으로 꺼져 있습니다. 켜면 **모델을 내려받느라 몇 분** 걸립니다.
- **8번에서 평가 결과가 예상과 반대로 나옵니다.** 오타가 아니니 그대로 읽어나가세요.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**를 적어두었습니다. 셀 뒤에는 두 가지가 붙습니다 —
  `show()`로 프로젝트 소스를 펼친 뒤에는 **코드에서 짚을 곳**이, 실제로 돌려본 뒤에는
  **결과 읽는 법**이 나옵니다.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](../../../troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| `AssertionError: 프로젝트 경로를 찾지 못했습니다` | **이 노트북은 실제 프로젝트 파일을 열어야 돌아갑니다.** Colab이라면 `git clone`이 실패했고, 로컬이라면 저장소 밖에서 노트북을 열었습니다 | Colab: 네트워크·프록시 확인 후 첫 셀 재실행. 로컬: 저장소를 통째로 받아 원래 폴더 구조 그대로 열기 |
| `show(...)`에서 `FileNotFoundError` | 파일명 오타이거나 프로젝트 구조가 바뀜 | `import os; print(os.listdir(SRC))`로 실제 파일 목록 확인 |
| import한 함수가 예전 동작을 한다 | 프로젝트 파일을 수정했지만 파이썬이 이미 불러둔 모듈을 재사용 | `import importlib; importlib.reload(모듈)` 또는 런타임 재시작 |
| `ModuleNotFoundError` — 프로젝트 모듈을 못 찾음 | `sys.path.insert(0, SRC)` 셀을 건너뜀 | 맨 위 준비 셀부터 순서대로 실행 |
| `리랭커를 건너뛰었습니다` | `TRY_RERANKER = False`가 기본값입니다 | 정상입니다. 돌려보려면 `True`로 바꾸세요 — **모델이 2GB라 Colab에서 몇 분** 걸립니다 |
| 리랭커 셀이 다운로드에서 멈춘 것 같다 | 모델을 처음 받는 중 | 정상입니다. 급하면 `TRY_RERANKER = False`로 되돌리세요 |
| OpenSearch 연결 에러 | 인프라 없이 돌도록 되어 있습니다 | 해당 코드는 `show()`로 읽기만 합니다 |
| 검색 결과가 기대와 다르다 | 임베딩 없이 TF-IDF로 대체된 상태 | `rag-pipeline-practice` 04번 실습 3의 "진짜 시험"에서 이 한계를 다룹니다 |
| `hit@5`·`MRR` 값이 본문과 다르다 | 평가셋·검색 방식이 조금만 달라도 값이 흔들립니다 | **절대값보다 "무엇을 재는 지표인지"** 를 보세요 |

여기 없는 문제(설치 실패, 한글 깨짐, API 키 설정 방법)는 [troubleshooting.md](../../../troubleshooting.md)에 모아뒀습니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q langchain-core langchain-text-splitters scikit-learn python-dotenv
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "rag-regulation-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-not-used-in-this-notebook")

for name in sorted(os.listdir(SRC)):
    path = os.path.join(SRC, name)
    print(f"  {name:<14} {len(open(path, encoding='utf-8').read().splitlines()):>3}줄")

| 파일 | 역할 | 외부 의존 |
|---|---|---|
| `parse.py` | 장>절>조>항 구조 파싱 + 무결성 검증 | **없음** (그래서 바로 돌려볼 수 있습니다) |
| `ingest.py` | 파싱 → 조항 청킹 → 임베딩 → 색인 | OpenSearch + OpenAI |
| `query.py` | 하이브리드 검색 → 리랭킹 → 답변 | OpenSearch + OpenAI |
| `evaluate.py` | 골든셋으로 검색 품질 채점 | OpenSearch |
| `api.py` | `/chat` 엔드포인트 | 위와 동일 |

`parse.py`가 아무 라이브러리도 안 쓴다는 게 우연이 아닙니다.
**문서 구조를 읽는 일은 검색엔진이나 LLM과 아무 상관이 없기 때문**입니다.
관심사가 분리되면 이렇게 따로 떼어 테스트할 수 있습니다.

## 1. 문제부터 — 1000자로 자르면 무슨 일이 벌어지나

먼저 이 프로젝트가 다루는 문서를 봅시다. 연습용 규정이 저장소에 들어 있습니다.

In [ ]:
regulation_path = os.path.join(PROJECT, "data", "sample_regulation.txt")
regulation_text = open(regulation_path, encoding="utf-8").read()

print(regulation_text[:600])
print("...")
print(f"\n전체 {len(regulation_text)}자")

이제 02에서 배운 방식대로 1000자로 잘라보겠습니다. 그리고 **제11조(재택근무)가 어떻게 되는지** 봅시다.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""]
)
fixed_chunks = splitter.split_documents([Document(page_content=regulation_text)])
print(f"1000자 청킹 결과: {len(fixed_chunks)}개 청크\n")

# "재택근무 중 연장근로 수당"의 답인 제11조 ③항이 어느 청크에 들어갔는지 찾아봅니다.
for i, chunk in enumerate(fixed_chunks):
    if "재택근무 중 발생한 연장근로" in chunk.page_content:
        print(f"답이 들어 있는 청크: #{i}")
        print("=" * 60)
        print(chunk.page_content[:400])
        print("=" * 60)
        break

**결과 읽는 법** — 이 청크를 잘 보세요. 답(`③ 재택근무 중 발생한 연장근로에 대하여는...`)은 들어 있습니다.
그런데 **이 조각이 무슨 조인지 알 수 있나요?**

경계가 어디서 갈렸는지에 따라 `제11조(재택근무)`라는 표지가 **앞 청크에 남아 있을 수 있습니다.**
그러면 이 조각에는 "재택근무"라는 단어도, 조항 번호도 없게 됩니다.

결과는 두 가지입니다.

1. **검색에 안 걸립니다** — "재택근무 수당"으로 물었는데 조각 안에 그 단어가 없으니까요
2. **걸려도 출처를 못 댑니다** — AI가 "제11조 ③항에 따르면"이라고 말할 근거가 조각에 없습니다

실제로 확인해봅시다. 각 청크가 조항 표지를 갖고 있는지 세어보겠습니다.

In [ ]:
import re

article_re = re.compile(r"제\s*\d+\s*조")

print("청크별 조항 표지 포함 여부:")
orphans = 0
for i, chunk in enumerate(fixed_chunks):
    head = chunk.page_content[:40].replace("\n", " ")
    has = bool(article_re.search(chunk.page_content[:60]))
    if not has:
        orphans += 1
    print(f"  #{i}: {'✓' if has else '✗ 시작부에 조 표지 없음'}  {head}...")

print(f"\n조 표지 없이 시작하는 청크: {orphans}/{len(fixed_chunks)}개")

> 💡 이 문서는 짧아서 청크가 몇 개 안 되지만, 실제 200페이지 규정집에서는
> **수십 개의 청크가 "자기가 무슨 조인지 모르는 상태"**가 됩니다.

## 2. `parse.py` — 문서의 진짜 구조를 읽기

해법은 단순합니다. **자로 재지 말고, 문서가 이미 갖고 있는 경계를 쓰면 됩니다.**

`parse.py`는 외부 의존이 없으니 바로 import해서 돌려볼 수 있습니다.

In [ ]:
show("parse.py", grep="^SECTION_RE|^ARTICLE_RE|^CIRCLED|^PARAGRAPH_RE")

**코드에서 짚을 곳** — 정규식 네 개가 전부입니다. 규정 문서는 형식이 정해져 있어서 이 정도로 상당히 정확하게 잡힙니다.

`ARTICLE_RE`를 뜯어보면:

```
제\s*(\d+)\s*조          "제11조"  -> 11
(?:\s*의\s*(\d+))?       "의2"     -> 2   (없을 수도 있어서 ? )
(?:\(([^)]*)\))?          "(재택근무)" -> 재택근무  (제목 없는 문서도 있어서 ? )
```

> 💡 **왜 LLM을 안 쓰고 정규식일까요?** 03에서 본 것처럼 LLM 정형 출력으로도 구조를 뽑을 수 있습니다.
> 하지만 조 번호는 **틀리면 안 되는 값**입니다. 규칙 기반은 100% 재현 가능하고, 비용이 0이고,
> 틀리면 어디가 틀렸는지 바로 보입니다. LLM은 셋 다 아닙니다.
> 형식이 제각각인 문서를 만나면 그때 LLM을 섞는 걸 고려하면 됩니다.

이제 실제로 파싱해봅시다.

In [ ]:
from parse import check_article_sequence, parse_articles, split_paragraphs

articles = parse_articles([regulation_text])
print(f"조 {len(articles)}개를 찾았습니다.\n")
for a in articles:
    print(f"  {a.full_path}  ({len(a.body)}자)")

**결과 읽는 법**

`제3장 근무 > 제11조(재택근무)` — **계층 경로**가 만들어졌습니다.
이게 뒤에서 청크마다 붙게 될 프리픽스입니다.

계층을 추적하는 부분이 이 파일에서 제일 까다로운 곳인데, 코드를 보면 이유가 보입니다.

In [ ]:
show("parse.py", grep="current_path = \\[entry|depth = |current_path.append|편장절관")

**코드에서 짚을 곳**

`제3장`을 만나면 **그보다 깊거나 같은 단계를 전부 버립니다.**
이렇게 하지 않으면 제3장으로 넘어갔는데도 제2장의 절이 경로에 남습니다.

깊이를 리스트 위치가 아니라 **값으로 들고 다니는** 이유도 여기 있습니다.
문서에 "편"이 없고 "장"부터 시작하는 경우가 많은데, 위치를 깊이로 쓰면 어긋나거든요.

## 3. 무결성 검증 — 조용한 실패를 잡아내기

이 프로젝트에서 제가 가장 중요하게 보는 함수입니다.

In [ ]:
show("parse.py", grep="def check_article_sequence|warnings|previous|missing|가지번호")

In [ ]:
for warning in check_article_sequence(articles):
    print("⚠️ ", warning)

**결과 읽는 법**

**제16조가 없다고 잡아냈습니다.**

PDF 텍스트 추출은 생각보다 자주 실패합니다. 2단 조판, 표 안에 들어간 조문, 이미지로 된 페이지…
문제는 **실패가 조용하다**는 겁니다. 에러도 안 나고, 그냥 그 조항만 없습니다.

검증이 없으면 이런 챗봇이 배포됩니다.

> 사용자: "제16조가 뭔가요?"
> 챗봇: "관련 규정을 찾을 수 없습니다."
> (실제로는 규정에 있는데 색인이 안 된 것)

조 번호가 1, 2, 3... 으로 이어진다는 **문서의 성질을 이용해서** 추출 실패를 자동 감지하는 겁니다.
데이터 파이프라인에서 이런 검증을 넣을 수 있는 지점을 찾는 게 중요합니다.

> 💡 위 경고는 진짜 버그가 아니라, 검증이 동작하는 걸 보여주려고 샘플 파일에서 일부러 뺀 것입니다.
> `가지번호(제11조의2)`가 연속성 검사에서 제외되는 것도 확인해두세요. 제11조 다음에 제11조의2가 와도
> 경고가 나지 않아야 정상입니다.

## 4. 조항 단위 청킹 — 프리픽스가 하는 일

In [ ]:
show("parse.py", grep="def split_paragraphs|def _strip_heading|max_chars|len\\(article.body\\)|PARAGRAPH_RE.split")

**코드에서 짚을 곳**

**짧은 조는 통째로 두고, `max_chars`를 넘을 때만 항 단위로 쪼갭니다.**
조가 곧 "하나의 완결된 규칙"이라 통째로 있어야 문맥이 살기 때문입니다.

`_strip_heading()`이 조 표지를 떼는 이유도 봐두세요. 프리픽스로 따로 붙일 거라
본문에 남겨두면 같은 문구가 두 번 들어갑니다.

이제 `ingest.py`가 이걸 어떻게 조립하는지 봅니다.

In [ ]:
show("ingest.py", grep="def articles_to_documents|page_content=f|\"path\"|\"article\"|\"paragraph\"|MAX_ARTICLE_CHARS =")

In [ ]:
from ingest import articles_to_documents

article_chunks = articles_to_documents(articles, "sample_regulation.txt")
print(f"조항 단위 청킹: {len(article_chunks)}개 청크\n")

for chunk in article_chunks:
    if chunk.metadata["article"] == "제11조":
        print("=" * 60)
        print(chunk.page_content)
        print("=" * 60)
        print("메타데이터:", chunk.metadata)

**결과 읽는 법** — 1번에서 본 조각과 비교해보세요.

- 첫 줄에 **`제3장 근무 > 제11조(재택근무)`** 가 있습니다 → "재택근무"로 검색하면 걸립니다
- 메타데이터에 **`article`, `paragraph`, `path`** 가 있습니다 → 출처를 조항으로 표시할 수 있습니다
- ①②③이 **한 조각 안에 다 있습니다** → 문맥이 안 잘렸습니다

긴 조는 항 단위로 쪼개지는지도 확인해봅시다. 이 샘플은 조가 짧아서
`MAX_ARTICLE_CHARS`를 낮춰서 동작을 보겠습니다.

In [ ]:
target = [a for a in articles if a.number == "제12조"][0]

print("기본값(900자)에서는 쪼개지지 않음:", len(split_paragraphs(target)), "조각")
print(f"(제12조는 {len(target.body)}자라서 900자를 안 넘습니다)\n")

print("max_chars=100으로 강제 분할:")
for marker, body in split_paragraphs(target, max_chars=100):
    print(f"  [{marker}] {body[:52]}...")

**결과 읽는 법** — 항 표시(`①②③④`)가 메타데이터로 남기 때문에, 출처를 **"제12조 ②"** 까지 정확히 댈 수 있습니다.

## 5. 정말 나아졌나 — 검색해서 비교하기

여기가 이 노트북의 핵심입니다. **"좋아진 것 같다"가 아니라 실제로 검색해봅니다.**

OpenSearch와 OpenAI 임베딩 대신 TF-IDF로 검색을 흉내 냅니다.
(원리는 같습니다 — 글을 벡터로 바꾸고 가까운 것을 찾는다.)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def make_searcher(chunks):
    """청크 목록을 받아 '질문 -> 상위 k개 청크' 검색 함수를 만들어 돌려준다."""
    corpus = [c.page_content for c in chunks]
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    matrix = vectorizer.fit_transform(corpus)

    def search(question, k=5):
        scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
        return [(chunks[i], scores[i]) for i in scores.argsort()[::-1][:k]]

    return search


# 두 가지 청킹 방식으로 각각 검색기를 만듭니다.
search_fixed = make_searcher(fixed_chunks)
search_article = make_searcher(article_chunks)

question = "재택근무 중에 야근하면 수당 받을 수 있나요?"

print(f"질문: {question}\n")
print("--- 1000자 고정 청킹 ---")
for chunk, score in search_fixed(question, k=2):
    print(f"  [{score:.3f}] {chunk.page_content[:70].replace(chr(10), ' ')}...")

print("\n--- 조항 단위 청킹 ---")
for chunk, score in search_article(question, k=2):
    label = chunk.metadata.get("article", "?")
    print(f"  [{score:.3f}] {label}: {chunk.page_content[:70].replace(chr(10), ' ')}...")

> 📝 **결과를 읽는 법**: 조항 단위 쪽은 **어느 조가 걸렸는지 바로 보입니다.**
> 고정 청킹 쪽은 조각이 커서 여러 조가 섞여 있고, 정확히 어느 조항이 근거인지 알 수 없습니다.
>
> `analyzer="char_wb"`를 쓴 이유도 알아두세요. 한국어는 조사가 붙어서 단어 단위 TF-IDF가
> 잘 안 맞습니다(02에서 본 `휴가를`/`휴가는` 문제). 글자 n-gram으로 하면 그 문제가 완화됩니다.
> 실제 프로젝트에서는 OpenSearch에 **노리(Nori) 형태소 분석기**를 붙여서 해결합니다.

## 6. 하이브리드 검색 — 왜 두 번 검색하나

`query.py`는 벡터 검색과 키워드 검색을 **둘 다** 돌려서 합칩니다. 이유가 있습니다.

In [ ]:
show("query.py", grep="def search_similar_docs|def search_keyword_docs|def reciprocal_rank_fusion|def search_hybrid_docs|scores\\[key\\]|key = \\(doc|RERANK_CANDIDATES = ")

| 검색 방식 | 강한 질문 | 약한 질문 |
|---|---|---|
| 벡터(의미) | "집에서 일할 때 야근수당" → 단어가 달라도 찾음 | "제12조" → 조항 번호는 의미가 없음 |
| 키워드(BM25) | "제12조" → 정확히 매칭 | "야근수당" → 규정엔 "연장근로 수당"이라 안 걸림 |

서로의 약점을 정확히 메웁니다. 문제는 **점수 스케일이 다르다**는 것 —
코사인 유사도(0~1)와 BM25(제한 없음)를 그냥 더할 수 없습니다.

그래서 **RRF(Reciprocal Rank Fusion)** 를 씁니다. 점수를 버리고 **등수만** 봅니다.

`1 / (k + 순위)` 를 검색별로 더하면, 두 검색 모두에서 상위에 있던 문서가 높은 점수를 받습니다.
실제 프로젝트 함수를 그대로 import해서 돌려봅시다.

In [ ]:
from query import reciprocal_rank_fusion

# 두 검색이 각각 다른 순서로 결과를 냈다고 가정합니다.
vector_results = [c for c, _ in search_article("집에서 일할 때 초과근무 수당", k=4)]
keyword_results = [c for c in article_chunks if "제12조" in c.metadata.get("article", "")][:1]
keyword_results += [c for c, _ in search_article("연장근로 가산 지급", k=3)]

print("벡터 검색 결과:")
for i, c in enumerate(vector_results, 1):
    print(f"  {i}위 {c.metadata['article']}")

print("\n키워드 검색 결과:")
for i, c in enumerate(keyword_results, 1):
    print(f"  {i}위 {c.metadata['article']}")

fused = reciprocal_rank_fusion(vector_results, keyword_results)
print("\nRRF 병합 결과:")
for i, c in enumerate(fused[:5], 1):
    print(f"  {i}위 {c.metadata['article']}")

> 💡 `reciprocal_rank_fusion`의 중복 제거 키를 보세요.
> `(source, page, page_content)` 세 개를 묶어서 씁니다. 본문만 키로 쓰면
> **우연히 내용이 같은 다른 문서**가 하나로 합쳐지면서 한쪽 출처가 통째로 사라집니다.
> 규정집 두 권에 같은 문구가 있는 건 흔한 일이라 실제로 일어납니다.

## 7. 리랭킹 — 빠른 검색 20개, 정확한 모델 4개

RRF까지 하면 후보 20개가 나옵니다. 여기서 한 번 더 줄을 세웁니다.

In [ ]:
show("query.py", grep="def rerank|bi-encoder|cross-encoder|따로따로|한 문장으로|model.predict|pairs = |USE_RERANKER|except Exception")

**코드에서 짚을 곳** — 왜 한 번 더 할까요? **계산 방식이 다르기 때문**입니다.

| | 임베딩 (bi-encoder) | 리랭커 (cross-encoder) |
|---|---|---|
| 방식 | 질문과 문서를 **따로** 벡터화 후 거리 계산 | 질문과 문서를 **붙여서** 모델에 통째로 입력 |
| 속도 | 문서 벡터를 미리 만들어둠 → 100만 건도 순식간 | 후보마다 모델 실행 → 느림 |
| 정확도 | 대략적 | 훨씬 정확 |

그래서 역할을 나눕니다. **빠른 검색으로 20개까지 좁히고, 정확한 리랭커로 4개를 고릅니다.**
사서가 서가에서 관련 있어 보이는 책 20권을 뽑아온 뒤, 목차를 펼쳐보고 4권을 고르는 것과 같습니다.

실무에서 검색 품질을 올릴 때 **비용 대비 효과가 가장 큰 단계가 보통 여기**입니다.

리랭커 모델은 2GB짜리라 이 노트북에서는 기본으로 받지 않습니다. 돌려보고 싶으면 아래 셀에서
`TRY_RERANKER = True`로 바꾸세요. (Colab 기준 몇 분 걸립니다.)

In [ ]:
TRY_RERANKER = False  # True로 바꾸면 실제 모델을 내려받아 돌려봅니다

if TRY_RERANKER:
    !pip install -q sentence-transformers
    from sentence_transformers import CrossEncoder

    model = CrossEncoder("BAAI/bge-reranker-base")  # 더 가벼운 쪽으로
    candidates = [c for c, _ in search_article(question, k=8)]
    pairs = [(question, c.page_content) for c in candidates]
    scores = model.predict(pairs)

    print(f"질문: {question}\n")
    print("리랭킹 후 순서:")
    for c, s in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True):
        print(f"  {s:>7.3f}  {c.metadata['article']} {c.metadata.get('path', '')}")
else:
    print("리랭커를 건너뛰었습니다. TRY_RERANKER = True 로 바꾸면 실제로 돌아갑니다.")
    print()
    print("참고: query.py의 rerank()는 모델 로딩에 실패해도 예외를 잡아서")
    print("      리랭킹만 건너뛰고 챗봇 자체는 계속 동작하게 되어 있습니다.")
    print("      '있으면 좋지만 없어도 서비스는 살아야 하는' 기능의 전형적인 처리 방식입니다.")

## 8. 평가 — "좋아진 것 같다"를 숫자로 바꾸기

여기까지 청킹을 바꾸고, 하이브리드를 붙이고, 리랭커를 달았습니다.
그런데 **정말 좋아진 걸까요?** 질문 몇 개 던져보고 판단하는 건 위험합니다.
어제 되던 게 오늘 안 되는 것도 눈치채지 못하고요.

그래서 정답지(골든셋)를 만들어두고 채점합니다.

In [ ]:
import json

golden_path = os.path.join(PROJECT, "data", "golden_set.json")
golden = json.loads(open(golden_path, encoding="utf-8").read())

print("골든셋 만드는 요령:")
for line in golden["_설명"]:
    if line:
        print("  ", line)

print(f"\n총 {len(golden['questions'])}문항:\n")
for q in golden["questions"][:5]:
    print(f"  {q['question']}")
    print(f"      -> 정답: {', '.join(q['articles'])}  ({q['type']})")

**결과 읽는 법**

**질문을 규정 문구 그대로 베끼지 않은 게 핵심**입니다.
"연차유급휴가는 며칠인가"라고 물으면 글자가 그대로 겹쳐서 항상 만점이 나옵니다.
그러면 개선 여부를 판단할 수 없죠. 실제 사람이 쓰는 말투로 써야 합니다.

이제 지표를 봅시다. `evaluate.py`의 함수들입니다.

In [ ]:
show("evaluate.py", grep="def hit_and_rank|def recall_at_k|for rank, doc|return True, rank|found = |K = ")

| 지표 | 의미 | 왜 보나 |
|---|---|---|
| **hit@5** | 상위 5개 안에 정답 조항이 있던 질문 비율 | 이게 낮으면 프롬프트를 아무리 다듬어도 소용없음 (근거가 안 넘어감) |
| **MRR** | 정답이 1등이면 1점, 2등이면 1/2점... 의 평균 | hit@5가 같아도 MRR이 높으면 정답을 더 위에 올려둔 것 |
| **recall@5** | 정답이 여러 개일 때 몇 개나 찾았는지 | 여러 조항을 봐야 답이 되는 질문용 |

직접 채점해봅시다. **1000자 청킹 vs 조항 청킹**을 같은 골든셋으로 비교합니다.

> ⚠️ 아래 결과는 예상과 다르게 나옵니다. **그게 이 절의 핵심입니다.** 숫자를 보고 놀라지 말고,
> 왜 그런지 같이 파고들겠습니다.

In [ ]:
def article_of(chunk):
    """청크가 어느 조에서 나왔는지 알아낸다.

    조항 청킹은 메타데이터에 있지만, 고정 길이 청킹은 없어서 본문에서 찾아야 한다.
    (이것 자체가 고정 길이 청킹의 약점입니다 — 출처를 조항으로 특정할 수가 없습니다.)
    """
    if chunk.metadata.get("article"):
        return {chunk.metadata["article"]}
    return set(re.findall(r"제\s*\d+\s*조(?:의\s*\d+)?", chunk.page_content))


def evaluate(search_fn, questions, k=5):
    hits = reciprocal = recall_sum = 0.0
    misses = []

    for item in questions:
        gold = set(item["articles"])
        results = search_fn(item["question"], k=k)

        rank_found = 0
        covered = set()
        for rank, (chunk, _score) in enumerate(results, 1):
            found = article_of(chunk) & gold
            covered |= found
            if found and not rank_found:
                rank_found = rank

        if rank_found:
            hits += 1
            reciprocal += 1 / rank_found
        else:
            misses.append(item["question"])
        recall_sum += len(covered) / len(gold)

    n = len(questions)
    return {"hit@5": hits / n, "MRR": reciprocal / n, "recall@5": recall_sum / n, "misses": misses}


questions = golden["questions"]
results = {
    "1000자 고정 청킹": evaluate(search_fixed, questions),
    "조항 단위 청킹": evaluate(search_article, questions),
}

print(f"{'청킹 방식':<20}{'hit@5':>10}{'MRR':>10}{'recall@5':>12}")
print("-" * 52)
for name, scores in results.items():
    print(f"{name:<20}{scores['hit@5']:>10.3f}{scores['MRR']:>10.3f}{scores['recall@5']:>12.3f}")

print("\n[놓친 질문]")
for name, scores in results.items():
    print(f"  {name}:")
    for q in scores["misses"] or ["없음"]:
        print(f"    - {q}")

## 8-1. 잠깐, 결과가 이상합니다

**1000자 고정 청킹이 전 항목 만점(1.000)입니다.** 조항 청킹보다 높습니다.

여기서 "아, 고정 청킹이 더 좋구나" 하고 넘어가면 안 됩니다.
**만점이 나왔다는 건 보통 지표가 고장 났다는 신호**입니다.

무엇이 잘못됐는지 찾아봅시다. 먼저 청크가 몇 개인지 보세요.

In [ ]:
print(f"고정 청킹 청크 수: {len(fixed_chunks)}개")
print(f"조항 청킹 청크 수: {len(article_chunks)}개")
print(f"\n채점에 쓴 k = 5")
print()
print("청크 하나에 조가 몇 개나 들어 있나:")
print("  고정 청킹:", [len(article_of(c)) for c in fixed_chunks])
print("  조항 청킹:", [len(article_of(c)) for c in article_chunks[:8]], "...")

**결과 읽는 법** — 찾았습니다. **고정 청킹은 청크가 3개뿐인데 상위 5개를 가져오라고 했습니다.**

`k`가 전체 청크 수보다 크면, 검색이 무슨 짓을 해도 **모든 청크가 결과에 들어갑니다.**
정답이 어딘가에는 있으니 hit@5는 **항상 1.000**입니다. 검색 성능을 전혀 재고 있지 않았던 겁니다.

게다가 청크 하나에 조가 5~10개씩 들어 있습니다.
"정답 조항을 찾았다"고 채점됐지만, 실제로는 **조 8개가 뭉쳐 있는 덩어리를 통째로 가져온 것**이죠.
시험에서 "답이 이 교과서 안에 있습니다"라고 답한 것과 같습니다. 틀린 말은 아니지만 쓸모가 없습니다.

## 8-2. 제대로 재기

두 가지를 고칩니다.

1. **`k`를 청크 수보다 작게** — 검색이 실제로 골라내게 만듭니다
2. **정밀도를 같이 봅니다** — 정답을 가져오면서 **쓸데없는 걸 얼마나 딸려 오는지**

In [ ]:
print(f"{'k':<4}{'고정 hit':>10}{'고정 MRR':>10}{'조항 hit':>10}{'조항 MRR':>10}")
print("-" * 44)
for k in (1, 2, 3, 5):
    a = evaluate(search_fixed, questions, k=k)
    b = evaluate(search_article, questions, k=k)
    print(f"{k:<4}{a['hit@5']:>10.3f}{a['MRR']:>10.3f}{b['hit@5']:>10.3f}{b['MRR']:>10.3f}")

print("\n(고정 청킹은 k를 아무리 낮춰도 1.000입니다. 청크 3개 중 하나만 뽑아도")
print(" 그 안에 조가 8개쯤 들어 있어서 정답이 '포함'되니까요.)")

In [ ]:
def noise_per_answer(search_fn, k=1):
    """정답을 얻기 위해 함께 딸려오는 조의 개수 (적을수록 정밀함)."""
    return sum(len(article_of(c)) for item in questions for c, _ in search_fn(item["question"], k=k)) / len(questions)


def chars_sent_to_llm(search_fn, k=1):
    """LLM에게 실제로 넘어가는 평균 글자 수 (곧 비용)."""
    return sum(len(c.page_content) for item in questions for c, _ in search_fn(item["question"], k=k)) / len(questions)


print(f"{'청킹 방식':<16}{'딸려오는 조':>12}{'LLM 입력 글자':>14}")
print("-" * 42)
for name, fn in (("1000자 고정", search_fixed), ("조항 단위", search_article)):
    print(f"{name:<16}{noise_per_answer(fn):>12.1f}{chars_sent_to_llm(fn):>14.0f}")

**결과 읽는 법**

**이제 진짜 차이가 보입니다.**

같은 질문에 답하기 위해 고정 청킹은 **조 7.8개, 907자**를 LLM에게 넘깁니다.
조항 청킹은 **조 1개, 194자**면 됩니다. 4~5배 차이입니다.

이게 실제로 의미하는 것:

- **비용** — LLM 입력 토큰이 4배 이상. 질문 10만 건이면 무시할 수 없습니다
- **정확도** — 관련 없는 조 7개가 같이 들어가면 AI가 엉뚱한 조를 근거로 답할 확률이 올라갑니다
- **출처** — 조가 8개 섞인 덩어리에서 AI는 "제11조 ③항에 따르면"이라고 말할 근거가 없습니다

**hit@5만 봤으면 이걸 전부 놓쳤을 겁니다.**

## 8-3. 여기서 진짜 배울 것

이 절에서 얻어야 할 교훈은 "조항 청킹이 이겼다"가 아닙니다.

> **지표는 만들었다고 끝이 아니다. 그 지표가 실제로 무엇을 재고 있는지 확인해야 한다.**

만점이 나오면 기뻐할 게 아니라 의심해야 합니다. 실무에서 평가 하네스를 만들 때
가장 흔한 실패가 **아무것도 재지 않는 지표를 만들어놓고 안심하는 것**입니다.

이번 경우의 체크리스트:

- [ ] `k`가 전체 문서 수보다 작은가? (아니면 hit@k는 항상 1.0)
- [ ] 골든셋 질문이 문서 문구를 그대로 베끼지 않았는가? (베끼면 항상 만점)
- [ ] 재고 싶은 게 recall인가 precision인가? 지금 지표가 그걸 재고 있는가?
- [ ] 점수가 다 같으면, 구분되는 다른 지표가 필요한 것은 아닌가?

> 💡 `evaluate.py`가 실제 OpenSearch에 대해 돌 때는 청크가 수백~수천 개라 `k=5`가 정상적으로
> 동작합니다. 여기서 문제가 된 건 **샘플 문서가 18개 조뿐**이기 때문입니다.
> 하지만 "왜 이 지표가 여기서는 무의미한가"를 아는 것과 모르는 것은 큰 차이입니다.

이제 파라미터를 바꿔가며 점수가 어떻게 움직이는지 봅시다. (이번엔 `k=2`로 재겠습니다.)

In [ ]:
print(f"{'설정':<22}{'hit@2':>9}{'MRR':>9}{'청크 수':>9}{'LLM 입력':>10}")
print("-" * 59)

for max_chars in (60, 100, 300, 900):
    docs = []
    for a in articles:
        for marker, body in split_paragraphs(a, max_chars=max_chars):
            docs.append(
                Document(
                    page_content=f"{a.full_path}\n{body}",
                    metadata={"source": "s.txt", "page": a.page, "article": a.number, "paragraph": marker},
                )
            )
    searcher = make_searcher(docs)
    s = evaluate(searcher, questions, k=2)
    print(
        f"max_chars={max_chars:<12}{s['hit@5']:>9.3f}{s['MRR']:>9.3f}{len(docs):>9}"
        f"{chars_sent_to_llm(searcher, k=2):>10.0f}자"
    )

> 📝 **읽는 법**: `max_chars`를 줄이면 조가 항 단위로 쪼개져 청크 수가 늘고 LLM 입력이 줄어듭니다.
> 대신 문맥이 잘려서 hit이 떨어질 수 있습니다. **정확도와 비용의 트레이드오프**가 숫자로 보이는 거죠.
>
> 어느 값이 최선인지는 문서와 질문에 따라 다릅니다. 중요한 건 **이제 그걸 감이 아니라
> 실험으로 정할 수 있다**는 것입니다. 이게 평가 하네스를 만드는 이유의 전부입니다.
>
> [놓친 질문] 목록도 계속 보세요. 점수 한 줄보다 실패 사례 하나가 고칠 거리를 더 많이 알려줍니다.

## 9. 프롬프트와 출처 — 마지막 한 걸음

검색이 끝나면 찾은 조각을 프롬프트에 넣습니다.

In [ ]:
show("query.py", grep="PROMPT_TEMPLATE = |아래 \\[관련 규정\\]|답변에는 근거|def format_citation|def build_prompt|context = |TOP_K = ")

**코드에서 짚을 곳** — 프롬프트에서 중요한 두 줄입니다.

> **"아래 [관련 규정]만 근거로 답변하고, 근거가 없으면 모른다고 답하세요."**
> **"답변에는 근거가 된 조항 번호를 반드시 함께 적으세요."**

첫 줄이 **그라운딩**입니다. 오픈북 시험에서 "교과서에 나온 내용만 쓰세요"라고 하는 것과 같습니다.
둘째 줄은 조항 청킹이 있어야 비로소 지킬 수 있는 요구입니다. 근거 조각에 조항 번호가 없으면
AI도 적을 수가 없으니까요. **1번에서 본 문제가 여기서 답변 품질로 이어집니다.**

출처 표시도 마찬가지입니다.

In [ ]:
from query import format_citation

for chunk in article_chunks[:3]:
    print(format_citation(chunk))

# 파서를 안 탄 문서(고정 길이 청킹)는 조항 정보가 없어서 예전처럼 페이지만 나옵니다.
fallback_doc = Document(page_content="...", metadata={"source": "notice.pdf", "page": 3})
print(format_citation(fallback_doc), " <- 조항 정보가 없는 경우")

**결과 읽는 법**

`p.4` 대신 `제11조 ③ (p.4)`. 규정을 다루는 사람은 "4페이지"가 아니라 "제11조 ③항"으로 말하고,
**개정판이 나오면 페이지는 밀려도 조항 번호는 그대로 남습니다.**

## 10. 전체 조립 — `api.py`

마지막으로 이 모든 게 어떻게 하나의 API가 되는지 봅니다.

In [ ]:
show("api.py", grep="class ChatRequest|class ChatResponse|@app|def chat|docs = |sources = |reply = |return ChatResponse")

**코드에서 짚을 곳**

`/chat` 하나가 전부입니다. 그런데 주석에 중요한 게 적혀 있습니다.

```python
docs = search_hybrid_docs(request.question)   # 검색 한 번
sources = [format_citation(doc) for doc in docs]
reply = answer_with_docs(request.question, docs)   # 같은 docs 재사용
```

`answer(question)`을 그냥 부르면 내부에서 검색을 **또** 합니다.
화면에 출처도 보여줘야 해서 검색 결과가 필요한데, 그러면 검색이 두 번 돌죠.
그래서 `answer_with_docs()`를 따로 만들어 뒀습니다. 리랭커까지 붙은 지금은
중복 검색 한 번의 비용이 꽤 큽니다.

## 정리

```
PDF/텍스트
  -> parse.py           장>절>조>항 구조 + 무결성 검증
  -> ingest.py          조항 단위 청킹 + 계층 경로 프리픽스 -> 색인
  -> query.py           하이브리드 검색(20) -> 리랭킹(4) -> 그라운딩 프롬프트
  -> evaluate.py        골든셋 채점 (hit@5 / MRR / recall@5)
  -> api.py             /chat
```

| 결정 | 이유 |
|---|---|
| 고정 길이 대신 조항 단위 청킹 | 문서가 이미 갖고 있는 경계를 쓰는 게 정확 |
| 계층 경로를 프리픽스로 | 조각만 떼어놔도 무슨 조인지 알 수 있게 |
| 구조 파싱은 정규식 | 조 번호는 틀리면 안 되는 값 — 재현 가능·무료·디버깅 쉬움 |
| 조 번호 연속성 검증 | 추출 실패가 조용히 지나가지 않게 |
| 벡터 + 키워드 하이브리드 | 서로의 약점을 정확히 메움 |
| 점수 대신 등수로 병합(RRF) | 스케일이 다른 점수는 더할 수 없음 |
| 20개 좁히고 4개 고르기 | 빠른 검색과 정확한 모델의 역할 분담 |
| 리랭커 실패해도 계속 동작 | 있으면 좋지만 없어도 서비스는 살아야 함 |
| 출처를 페이지가 아니라 조항으로 | 사람이 그렇게 말하고, 개정에도 안 밀림 |
| 골든셋 + 지표 | 개선을 감이 아니라 숫자로 판단 |
| **지표 자체를 검증** | 만점이 나오면 기뻐할 게 아니라 의심 — 8-1에서 겪은 것 |

**가장 기억할 것**: **RAG의 품질은 대부분 검색 단계에서 결정되고,
검색이 좋아졌는지는 지표로만 알 수 있으며, 그 지표도 틀릴 수 있습니다.**
LLM을 더 비싼 걸로 바꾸기 전에 이 셋을 먼저 점검하세요.

**스스로 확인해보기**

- [ ] 고정 길이 청킹이 규정 문서에서 왜 불리한지 예를 들어 설명할 수 있다
- [ ] 계층 경로를 프리픽스로 붙이면 무엇이 좋아지는지 두 가지 말할 수 있다
- [ ] 조 번호 연속성 검증이 잡아내는 실패가 어떤 종류인지 안다
- [ ] 벡터 검색이 약한 질문과 키워드 검색이 약한 질문을 각각 들 수 있다
- [ ] RRF가 점수를 안 쓰고 등수를 쓰는 이유를 안다
- [ ] 리랭커를 20개 후보에만 돌리는 이유를 설명할 수 있다
- [ ] hit@k가 항상 1.0이 나오는 조건을 안다
- [ ] 출처를 페이지가 아니라 조항으로 표시하는 이유를 두 가지 말할 수 있다

## 연습 문제

**1. 골든셋 늘리기**
12문항은 예시 수준입니다. `data/golden_set.json`에 5문항을 추가해보세요.
단, **일부러 어렵게** 만드세요 — 규정 문구를 베끼지 말고, 여러 조항을 봐야 답이 되는 질문으로요.
추가한 뒤 8번의 평가를 다시 돌리면 점수가 어떻게 변하나요?

**2. 상호참조(xref) 확장**
제11조 ③항에는 "제12조에 따른 수당"이라고 적혀 있습니다. 제11조를 찾았을 때
**거기서 언급된 제12조를 자동으로 같이 가져오면** 답변이 완성됩니다.
`parse.py`에 본문에서 "제N조"를 뽑는 함수를 추가하고, 검색 결과를 1홉 확장해보세요.
확장 전후로 `recall@5`를 비교해보면 효과가 보입니다.

**3. 파서를 깨뜨려보기**
`parse_articles()`에 이런 텍스트를 넣으면 어떻게 되나요?
- `제 11 조 (재택근무)` — 공백이 들어간 경우
- 표 안에 들어가서 `제11조` 앞에 `|`가 붙은 경우
- 조 제목이 두 줄로 잘린 경우

깨지는 케이스를 찾아 정규식을 고쳐보세요. **어디까지 규칙으로 감당하고 어디부터 포기할지**를
정하는 것도 설계입니다.

**4. 고장 난 지표 또 찾아내기**
8-1에서 `k`가 청크 수보다 커서 hit@k가 항상 1.0이 되는 걸 봤습니다.
그럼 이런 경우는 어떨까요?
- 골든셋의 정답 조항이 전부 문서 앞부분에 몰려 있다면?
- 검색기가 항상 같은 청크 5개만 돌려준다면 MRR은 어떻게 나올까?

**"이 지표를 속이려면 어떻게 해야 하나"**를 생각해보면 지표의 빈틈이 보입니다.

**5. 답변 채점(faithfulness)**
`evaluate.py`는 "근거를 찾아왔는가"까지만 잽니다. "찾아온 근거대로 답했는가"는 별개입니다.
답변과 근거를 LLM에게 같이 주고 "이 답변이 근거에서 나왔는지" 판정시키는 평가를 설계해보세요.
어떤 함정이 있을까요? (힌트: 채점하는 LLM도 틀립니다)

**해설/정답**: [04_rag_regulation_solutions.ipynb](04_rag_regulation_solutions.ipynb)

## 시리즈를 마치며

네 개 프로젝트를 다 따라왔습니다.

| | 프로젝트 | 핵심 |
|---|---|---|
| 01 | `crawl-storage-example` | 원본은 손대지 말고 그대로 보관 |
| 02 | `preprocess-example` | 형식 통일 → 정제 → 청킹 |
| 03 | `document-input-example` | LLM에게 양식을 강제하고, 검증으로 안전망 |
| 04 | `rag-regulation-example` | 문서 구조를 살려서 검색하고, 숫자로 확인 |

관통하는 원칙이 하나 있습니다. **문제를 가장 이른 시점에 드러나게 하라.**
`config.py`의 시작 시점 검증, Pydantic 스키마, 조 번호 무결성 검증, 평가 하네스 —
전부 같은 이야기입니다. 조용한 실패가 제일 비쌉니다.

이어서 볼 것:
- [`rag-pipeline-practice/05_prompt_injection_defense`](../../rag-pipeline-practice/05_prompt_injection_defense/05_prompt_injection_defense.ipynb)
  — 검색된 문서를 프롬프트에 그대로 붙이는 구조의 보안 취약점과 방어
- 프로젝트 README의 "다음 단계" — 표·수식 처리, xref 확장, faithfulness 평가